Step 1 — Housekeeping: Import Libraries and Set File Paths

In [1]:
import sqlite3

import csv

import urllib.request

import os

 

In [2]:
BASE_URL = "https://raw.githubusercontent.com/rayj1981/cis3120/main/Module%207/"

BOOK_URL   = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL   = BASE_URL + "Loan.csv"

BOOK_PATH   = "Book.csv"
MEMBER_PATH = "Member.csv"
LOAN_PATH   = "Loan.csv"

DB_PATH = "library.db"

Step 2 — Download the CSV Files from GitHub

In [3]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:

    urllib.request.urlretrieve(url, path)

    print(f"Downloaded: {path}")

Downloaded: Book.csv
Downloaded: Member.csv
Downloaded: Loan.csv


Step 3 — Connect to the Database and Create Tables

In [4]:
conn = sqlite3.connect(DB_PATH)

conn.execute('PRAGMA foreign_keys = ON;')


 # CREATE TABLE statements
conn.executescript("""
CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
);

CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);

CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id)     REFERENCES Member(id)
);
""")



conn.commit()

print("Tables created.")

Tables created.


In [5]:
conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()

[('Book',), ('Loan',), ('Member',)]

Step 4 — Load Book.csv into the Book Table
Step 5 — Load Member.csv into the Member Table
Step 6 — Load Loan.csv into the Loan Table

In [6]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:

        conn.execute(

            'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',

            (row['callNo'], row['title'], row['author'])
        )

conn.commit()

print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])


with open(MEMBER_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:

        conn.execute(

            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',

            (int(row['id']), row['firstname'], row['lastName'])

        )

 

conn.commit()

print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])


with open(LOAN_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:

        # Convert empty dateReturned to None (→ NULL in SQLite)

        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

 

        conn.execute(

            '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)

               VALUES (?, ?, ?, ?, ?);''',

            (row['callNo'], int(row['id']),

             row['dateBorrowed'], date_returned, row['dateDue'])

        )

 

conn.commit()

print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Book rows loaded: 11
Member rows loaded: 4
Loan rows loaded: 4


5. Required SQL Queries

Query 1 — All Books

In [7]:
query = '''
SELECT * 
FROM Book 
ORDER BY author;'''

for row in conn.execute(query):
    print(row)


('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


Query 2 — Books Not Yet Returned

In [8]:
query = '''
SELECT title, firstname, lastName
FROM Loan
JOIN Member ON Loan.Id = Member.Id
Join Book ON Loan.callNo = Book.callNo
WHERE dateReturned is NULL;'''

for row in conn.execute(query):
    print(row)


("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


Query 3 — Loan History for a Specific Book

In [9]:
query = '''
SELECT firstname, lastName, dateBorrowed, dateDue, dateReturned
FROM Loan
JOIN Member ON Loan.Id = Member.Id
WHERE callNo = 'R 141 E45 2006'
ORDER BY dateBorrowed;'''



for row in conn.execute(query):
    print(row)


('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


Query 4 — Members Who Have Never Borrowed a Book

In [10]:
query = '''
SELECT Member.id, firstname, lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL;'''



for row in conn.execute(query):
    print(row)

(4, 'John', 'Martin')


Query 5 — Count of Loans per Member

In [11]:
query = '''

SELECT COUNT(Loan.id) AS loan_count, firstname, lastName
FROM Member 
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id
ORDER BY loan_count DESC;
'''


for row in conn.execute(query):
    print(row)



(2, 'David', 'Martin')
(1, 'John', 'Smith')
(1, 'Betty', 'Freeman')
(0, 'John', 'Martin')


Query 6 — Free-Choice Analytical Query
Which books have been borrowed more than once? This is importnat as it tell us the  books people acutally want to loan

In [12]:
query = '''

SELECT Book.title, COUNT(Book.callNo) AS NumLoans
FROM Book 
JOIN Loan ON Book.callNo = Loan.callNo
GROUP BY Book.title
HAVING NumLoans > 1
ORDER BY NumLoans DESC;
'''


for row in conn.execute(query):
    print(row)

('Medieval medicine and the plague', 2)


 brief Markdown summary (3–5 sentences) at the end describing one data quality observation you made and one limitation of this dataset for a real library system

One data quailty I made was that many of the books we not returned and this made date returned empty for most of the loans. A limitation of this dataset as to compared ot a real libaray system is how little information we have here it only contains a handful of members and books. In a real libaray sysstem there are books dated back to the 1900s and they have large amounts of data on loans and members.